# S11 - Proyecto Sello LLSW3: BI/ML distribuido sobre `red-telemetria`

**Equipo:** LLSW3 · **Estudiante:** Ramos Quispe, Jhan Logan · **Rol:** Streaming/Kafka

Mismo cierre de BI/ML de la guía, aplicado al sistema de monitoreo de tráfico de red del campus:

- **Parte A (batch):** KPIs de negocio sobre el Parquet de observabilidad propio de S9 (`red_telemetria_observabilidad`), publicados **una vez** en el Pushgateway.
- **Parte B (en vivo):** el `PipelineModel` de tráfico t+1 que S10 guardó (`modelo_red_rx_t1`) se redespliega **sin reentrenar** contra `red-telemetria`, y cada micro-lote publica en el Pushgateway la lectura real, la predicción y el error de la predicción anterior (Regla #37 en vivo).


In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = (
    SparkSession.builder
    .appName("s11-proyecto-llsw3-red")
    .master("local[*]")
    .config("spark.jars.packages", "org.apache.spark:spark-sql-kafka-0-10_2.13:4.2.0")
    .config("spark.sql.shuffle.partitions", "8")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")
spark.version

:: loading settings :: url = jar:file:/usr/local/lib/python3.10/site-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /root/.ivy2.5.2/cache
The jars for the packages stored in: /root/.ivy2.5.2/jars
org.apache.spark#spark-sql-kafka-0-10_2.13 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-df876924-c47b-4133-8cc5-b46068d79c82;1.0
	confs: [default]
	found org.apache.spark#spark-sql-kafka-0-10_2.13;4.2.0 in central


	found org.apache.spark#spark-token-provider-kafka-0-10_2.13;4.2.0 in central
	found org.apache.kafka#kafka-clients;3.9.2 in central
	found at.yawk.lz4#lz4-java;1.11.0 in central
	found org.xerial.snappy#snappy-java;1.1.10.8 in central
	found org.slf4j#slf4j-api;2.0.17 in central
	found org.apache.hadoop#hadoop-client-runtime;3.5.0 in central
	found org.apache.hadoop#hadoop-client-api;3.5.0 in central
	found com.google.code.findbugs#jsr305;3.0.0 in central
	found org.scala-lang.modules#scala-parallel-collections_2.13;1.2.0 in central
	found org.apache.commons#commons-pool2;2.13.1 in central
:: resolution report :: resolve 333ms :: artifacts dl 10ms
	:: modules in use:
	at.yawk.lz4#lz4-java;1.11.0 from central in [default]
	com.google.code.findbugs#jsr305;3.0.0 from central in [default]
	org.apache.commons#commons-pool2;2.13.1 from central in [default]
	org.apache.hadoop#hadoop-client-api;3.5.0 from central in [default]
	org.apache.hadoop#hadoop-client-runtime;3.5.0 from central in [def

:: retrieving :: org.apache.spark#spark-submit-parent-df876924-c47b-4133-8cc5-b46068d79c82
	confs: [default]
	0 artifacts copied, 11 already retrieved (0kB/8ms)
26/10/24 19:13:40 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


'4.2.0'

## Parte A — KPIs de negocio del flujo de red (batch)

Evidencia persistida por el notebook de proyecto de S9: cada lectura del agente de red con `isValid`, `processedAt` y `latencyMs`.


In [2]:
RUTA_OBS_S9 = "../s09-observabilidad-pipeline/artifacts/red_telemetria_observabilidad"
obs = spark.read.parquet(RUTA_OBS_S9)
print(f"Lecturas persistidas en S9: {obs.count():,}")
obs.groupBy("sensorId").agg(F.count("*").alias("lecturas"), F.min("kafkaTimestamp").alias("desde"),
                            F.max("kafkaTimestamp").alias("hasta")).show(truncate=False)

Lecturas persistidas en S9: 164


+-------------+--------+-----------------------+-----------------------+
|sensorId     |lecturas|desde                  |hasta                  |
+-------------+--------+-----------------------+-----------------------+
|llsw3-wi-fi-2|164     |2026-10-13 14:52:54.736|2026-10-13 14:55:54.907|
+-------------+--------+-----------------------+-----------------------+



In [3]:
kpi = obs.agg(
    F.count("*").alias("lecturas"),
    F.avg(F.col("isValid").cast("int")).alias("ratioValidas"),
    F.avg("latencyMs").alias("latenciaPromMs"),
    (F.avg("bytesRecibidosSeg") / 1e6).alias("rxPromMBps"),
    (F.max("bytesRecibidosSeg") / 1e6).alias("rxPicoMBps"),
    F.avg(F.col("bytesRecibidosSeg") / (F.col("velocidadEnlaceMbps") * 125000)).alias("utilizacionEnlace"),
).first()

print(f"Lecturas procesadas:          {kpi['lecturas']}")
print(f"Ratio de lecturas validas:    {kpi['ratioValidas']:.4f}")
print(f"Latencia promedio:            {kpi['latenciaPromMs']:.2f} ms")
print(f"Trafico recibido promedio:    {kpi['rxPromMBps']:.3f} MB/s")
print(f"Pico de trafico recibido:     {kpi['rxPicoMBps']:.3f} MB/s")
print(f"Utilizacion media del enlace: {kpi['utilizacionEnlace']*100:.3f} %")

Lecturas procesadas:          164
Ratio de lecturas validas:    1.0000
Latencia promedio:            2439.20 ms
Trafico recibido promedio:    0.323 MB/s
Pico de trafico recibido:     3.900 MB/s
Utilizacion media del enlace: 1.799 %


Se publican **una sola vez** con `PUT` en el grupo `job=llsw3_bi_red`, `instance=s11_batch`: el notebook termina y Prometheus no podría *scrapearlo* (2.3).


In [4]:
import urllib.request, urllib.parse, json, time

metricas_red = {
    "llsw3_bi_lecturas_total": kpi["lecturas"],
    "llsw3_bi_lecturas_validas_ratio": kpi["ratioValidas"],
    "llsw3_bi_latencia_promedio_ms": kpi["latenciaPromMs"],
    "llsw3_bi_rx_promedio_bytes": kpi["rxPromMBps"] * 1e6,
    "llsw3_bi_rx_pico_bytes": kpi["rxPicoMBps"] * 1e6,
    "llsw3_bi_utilizacion_enlace_ratio": kpi["utilizacionEnlace"],
}
cuerpo = "".join(f"# TYPE {n} gauge\n{n} {v}\n" for n, v in metricas_red.items())
URL_BATCH = "http://pushgateway:9091/metrics/job/llsw3_bi_red/instance/s11_batch"
with urllib.request.urlopen(urllib.request.Request(URL_BATCH, data=cuerpo.encode(), method="PUT")) as r:
    print("Pushgateway respondio:", r.status)

def consultar(expr):
    res = json.load(urllib.request.urlopen("http://prometheus:9090/api/v1/query?query=" + urllib.parse.quote(expr)))["data"]["result"]
    return res[0]["value"][1] if res else None

print("inmediatamente despues del PUT:", consultar("llsw3_bi_lecturas_total"))
time.sleep(20)
for n in metricas_red:
    print(f"{n}: {consultar(n)}")

Pushgateway respondio: 200
inmediatamente despues del PUT: None


llsw3_bi_lecturas_total: 164
llsw3_bi_lecturas_validas_ratio: 1
llsw3_bi_latencia_promedio_ms: 2439.201219512195
llsw3_bi_rx_promedio_bytes: 323096.9731707317
llsw3_bi_rx_pico_bytes: 3900360
llsw3_bi_utilizacion_enlace_ratio: 0.017986531611944963


## Parte B — Modelo de S10 redesplegado en vivo

Se recarga el `PipelineModel` de tráfico t+1 que S10 guardó — **sin reentrenar** — y se aplica a cada lectura de `red-telemetria` según llega, sin ventanas ni estado de Spark.


In [5]:
from pyspark.ml import PipelineModel
RUTA_MODELO_S10 = "../s10-series-tiempo-inferencia-streaming/artifacts/models/modelo_red_rx_t1"
modelo_red = PipelineModel.load(RUTA_MODELO_S10)
print("Modelo cargado desde:", RUTA_MODELO_S10)
print("etapas:", [type(s).__name__ for s in modelo_red.stages])

Modelo cargado desde: ../s10-series-tiempo-inferencia-streaming/artifacts/models/modelo_red_rx_t1
etapas: ['VectorAssembler', 'LinearRegressionModel']


In [6]:
from pyspark.sql.types import StructType, StringType, DoubleType, LongType

esquema_red = (
    StructType()
    .add("tipoEvento", StringType()).add("sensorId", StringType()).add("interfaz", StringType())
    .add("tipoInterfaz", StringType()).add("velocidadEnlaceMbps", LongType())
    .add("bytesRecibidosSeg", DoubleType()).add("bytesEnviadosSeg", DoubleType())
    .add("paquetesRecibidosSeg", DoubleType()).add("paquetesEnviadosSeg", DoubleType())
    .add("origen", StringType()).add("timestamp", LongType())
)
lecturas = (
    spark.readStream.format("kafka")
    .option("kafka.bootstrap.servers", "kafka:9092")
    .option("subscribe", "red-telemetria")
    .option("startingOffsets", "latest")
    .option("failOnDataLoss", "false")
    .load()
    .select(F.from_json(F.col("value").cast("string"), esquema_red).alias("e")).select("e.*")
    .where((F.col("tipoEvento") == "red.lectura") & (F.col("tipoInterfaz") == "fisica"))
)
predicciones_red = modelo_red.transform(lecturas)   # sin withWatermark ni window: una lectura basta

Cada micro-lote guarda sus predicciones en Parquet **y** hace un `PUT` al Pushgateway por tarjeta (`instance=sensorId`). Además publica el **error absoluto de la predicción anterior** frente a la lectura real que acaba de llegar: la comparación "datos en vivo" de la Regla #37, medida mientras ocurre.


In [7]:
RUTA_PRED = "./artifacts/predicciones_red_stream"
CHK_PRED = "./artifacts/chk_predicciones_red_stream"
PUSH_RED = "http://pushgateway:9091/metrics/job/llsw3_ml_red/instance"
ultima_prediccion = {}

def publicar_en_vivo(sensor, rx_real, prediccion, error):
    cuerpo = (f"# TYPE llsw3_ml_rx_real gauge\nllsw3_ml_rx_real {rx_real}\n"
              f"# TYPE llsw3_ml_rx_prediccion_siguiente gauge\nllsw3_ml_rx_prediccion_siguiente {prediccion}\n")
    if error is not None:
        cuerpo += f"# TYPE llsw3_ml_error_abs_vivo gauge\nllsw3_ml_error_abs_vivo {error}\n"
    urllib.request.urlopen(urllib.request.Request(f"{PUSH_RED}/{sensor}", data=cuerpo.encode(), method="PUT")).close()

def publicar_microlote(df, id_lote):
    filas = df.count()
    if filas == 0:
        print(f"Micro-lote {id_lote}: sin filas")
        return
    df.select(F.lit(id_lote).alias("batchId"), "sensorId", "timestamp",
              (F.col("timestamp") / 1000).cast("timestamp").alias("fechaHora"), "bytesRecibidosSeg",
              F.col("prediction").alias("prediccionRxSiguiente")).write.mode("append").parquet(RUTA_PRED)
    for f in df.select("sensorId", "timestamp", "bytesRecibidosSeg", "prediction").orderBy("timestamp").collect():
        previa = ultima_prediccion.get(f["sensorId"])
        error = abs(previa - f["bytesRecibidosSeg"]) if previa is not None else None
        publicar_en_vivo(f["sensorId"], f["bytesRecibidosSeg"], f["prediction"], error)
        ultima_prediccion[f["sensorId"]] = f["prediction"]
    print(f"Micro-lote {id_lote}: {filas} publicada(s) | real {f['bytesRecibidosSeg']:>11,.0f} B/s"
          f" | prediccion t+1 {f['prediction']:>11,.0f} B/s | error de la anterior {error if error is None else round(error)}")

consulta_red = (predicciones_red.writeStream.foreachBatch(publicar_microlote)
                .option("checkpointLocation", CHK_PRED).start())

Para que haya tráfico real que predecir, el nodo de captura descarga por ráfagas (20 s descargando, 20 s en reposo) mientras la consulta corre.


In [8]:
import threading

def rafagas(total_s=420, on=20, off=20):
    fin = time.time() + total_s
    while time.time() < fin:
        t = time.time() + on
        try:
            with urllib.request.urlopen("https://proof.ovh.net/files/1Gb.dat", timeout=10) as r:
                while time.time() < t and r.read(1 << 16):
                    pass
        except Exception as ex:
            print("descarga:", ex)
        time.sleep(off)

threading.Thread(target=rafagas, daemon=True).start()
time.sleep(60)

Micro-lote 0: sin filas


netlib-blas: JNI_OnLoad: dlopen(libblas.so.3) failed: libblas.so.3: cannot open shared object file: No such file or directory


Micro-lote 1: 1 publicada(s) | real      43,936 B/s | prediccion t+1     541,625 B/s | error de la anterior None


Micro-lote 2: 3 publicada(s) | real   9,496,685 B/s | prediccion t+1   5,485,783 B/s | error de la anterior 6408876


Micro-lote 3: 3 publicada(s) | real  17,074,348 B/s | prediccion t+1  10,664,748 B/s | error de la anterior 6090510


...

Micro-lote 21: 2 publicada(s) | real  18,969,802 B/s | prediccion t+1  12,953,920 B/s | error de la anterior 7476119


Micro-lote 22: 2 publicada(s) | real  20,118,756 B/s | prediccion t+1  12,506,233 B/s | error de la anterior 9967129


Micro-lote 23: 3 publicada(s) | real     362,993 B/s | prediccion t+1     715,696 B/s | error de la anterior 434982


Micro-lote 24: 2 publicada(s) | real       9,002 B/s | prediccion t+1     508,858 B/s | error de la anterior 636860


Micro-lote 25: 2 publicada(s) | real         661 B/s | prediccion t+1     506,392 B/s | error de la anterior 516037


### Confirmación con PromQL: la misma consulta, separada en el tiempo


In [9]:
from datetime import datetime
for i in range(3):
    print(datetime.now().strftime("%H:%M:%S"), "| llsw3_ml_rx_real =", consultar("llsw3_ml_rx_real"),
          "| prediccion =", consultar("llsw3_ml_rx_prediccion_siguiente"), "| error vivo =", consultar("llsw3_ml_error_abs_vivo"))
    if i < 2:
        time.sleep(20)

serie = json.load(urllib.request.urlopen("http://prometheus:9090/api/v1/query?query=" +
                                         urllib.parse.quote("llsw3_ml_rx_real[3m]")))["data"]["result"][0]["values"]
print("muestras de Prometheus en los ultimos 3 min:", len(serie), "| valores distintos:", len({v for _, v in serie}))

19:15:13 | llsw3_ml_rx_real = 18969802.1 | prediccion = 12953919.910274329 | error vivo = 7476119.052600529


Micro-lote 26: 2 publicada(s) | real   9,317,836 B/s | prediccion t+1   5,854,234 B/s | error de la anterior 8814278


Micro-lote 27: 2 publicada(s) | real       7,513 B/s | prediccion t+1     543,908 B/s | error de la anterior 487509


Micro-lote 28: 2 publicada(s) | real       8,297 B/s | prediccion t+1     507,598 B/s | error de la anterior 487866


Micro-lote 29: 2 publicada(s) | real      21,780 B/s | prediccion t+1     540,458 B/s | error de la anterior 499574


...

Micro-lote 39: 3 publicada(s) | real   2,725,154 B/s | prediccion t+1   2,348,580 B/s | error de la anterior 956782


Micro-lote 40: 2 publicada(s) | real   4,105,137 B/s | prediccion t+1   3,161,120 B/s | error de la anterior 1289321


Micro-lote 41: 2 publicada(s) | real   6,176,735 B/s | prediccion t+1   5,449,637 B/s | error de la anterior 1615358


Micro-lote 42: 2 publicada(s) | real   1,405,154 B/s | prediccion t+1   1,541,373 B/s | error de la anterior 4028646


19:15:53 | llsw3_ml_rx_real = 4105136.7 | prediccion = 3161119.907310779 | error vivo = 1289321.2411311334
muestras de Prometheus en los ultimos 3 min: 7 | valores distintos: 7


Con la consulta activa se arma y captura el tablero de Grafana del proyecto.


In [10]:
time.sleep(240)
print("micro-lotes procesados:", len(consulta_red.recentProgress))

Micro-lote 43: 2 publicada(s) | real   5,068,531 B/s | prediccion t+1   4,181,273 B/s | error de la anterior 1948171


Micro-lote 44: 3 publicada(s) | real   5,947,954 B/s | prediccion t+1   4,201,211 B/s | error de la anterior 376954


Micro-lote 45: 2 publicada(s) | real   8,403,848 B/s | prediccion t+1   6,588,120 B/s | error de la anterior 1853963


Micro-lote 46: 2 publicada(s) | real   5,807,657 B/s | prediccion t+1   3,656,308 B/s | error de la anterior 4148280


Micro-lote 47: 2 publicada(s) | real       4,242 B/s | prediccion t+1     500,973 B/s | error de la anterior 492537


...

Micro-lote 143: 3 publicada(s) | real       2,532 B/s | prediccion t+1     502,388 B/s | error de la anterior 497350


Micro-lote 144: 2 publicada(s) | real       1,343 B/s | prediccion t+1     497,898 B/s | error de la anterior 502887


Micro-lote 145: 2 publicada(s) | real       1,968 B/s | prediccion t+1     517,907 B/s | error de la anterior 497462


Micro-lote 146: 2 publicada(s) | real       2,282 B/s | prediccion t+1     510,360 B/s | error de la anterior 512398


micro-lotes procesados: 99


## Auditoría al detener la consulta (Regla #37)

Cada predicción se une con la lectura real que llegó **después** para la misma tarjeta (`lead`), y el error en producción se compara con el que S10 midió en la prueba.


In [11]:
consulta_red.stop()
from pyspark.sql import Window
pred = spark.read.parquet(RUTA_PRED)
w = Window.partitionBy("sensorId").orderBy("timestamp")
auditoria = (pred.withColumn("rxRealSiguiente", F.lead("bytesRecibidosSeg").over(w))
             .where(F.col("rxRealSiguiente").isNotNull())
             .withColumn("errorModelo", F.abs(F.col("prediccionRxSiguiente") - F.col("rxRealSiguiente")))
             .withColumn("errorPersistencia", F.abs(F.col("bytesRecibidosSeg") - F.col("rxRealSiguiente"))))
auditoria.select("batchId", "sensorId", "fechaHora", "bytesRecibidosSeg", "prediccionRxSiguiente",
                 "rxRealSiguiente", "errorModelo").orderBy("fechaHora").show(15, truncate=False)
r = auditoria.agg(F.count("*").alias("n"), F.avg("errorModelo").alias("mae"), F.avg("errorPersistencia").alias("maeP")).first()
print(f"Predicciones auditadas:         {r['n']}")
print(f"MAE del modelo en produccion:   {r['mae']:,.0f} B/s   (MAE en la prueba de S10: 113,202 B/s)")
print(f"MAE de la persistencia en vivo: {r['maeP']:,.0f} B/s")

Exception in thread "stream execution thread for [id = eb29c7ac-6d1a-4418-8456-1f4411a6dda2, runId = dc172579-2372-478b-8ce5-606eb5c880cd]" java.lang.StackOverflowError
	at java.base/java.util.regex.Pattern$Branch.match(Pattern.java:4914)
	at java.base/java.util.regex.Pattern$GroupHead.match(Pattern.java:4969)
	at java.base/java.util.regex.Pattern$Loop.match(Pattern.java:5078)
	at java.base/java.util.regex.Pattern$GroupTail.match(Pattern.java:5000)
	at java.base/java.util.regex.Pattern$BranchConn.match(Pattern.java:4878)
	at java.base/java.util.regex.Pattern$CharProperty.match(Pattern.java:4110)
	at java.base/java.util.regex.Pattern$Branch.match(Pattern.java:4914)
	at java.base/java.util.regex.Pattern$GroupHead.match(Pattern.java:4969)
	at java.base/java.util.regex.Pattern$Loop.match(Pattern.java:5078)
	at java.base/java.util.regex.Pattern$GroupTail.match(Pattern.java:5000)
	at java.base/java.util.regex.Pattern$BranchConn.match(Pattern.java:4878)
	at java.base/java.util.regex.Pattern$C

+-------+----------------+-----------------------+-----------------+---------------------+---------------+--------------------+
|batchId|sensorId        |fechaHora              |bytesRecibidosSeg|prediccionRxSiguiente|rxRealSiguiente|errorModelo         |
+-------+----------------+-----------------------+-----------------+---------------------+---------------+--------------------+
|1      |llsw3-ethernet-2|2026-10-24 19:14:14.34 |43936.0          |541625.2967810158    |1028468.0      |486842.7032189842   |
|2      |llsw3-ethernet-2|2026-10-24 19:14:15.363|1028468.0        |1510990.7481705807   |3907509.4      |2396518.651829419   |
|2      |llsw3-ethernet-2|2026-10-24 19:14:16.399|3907509.4        |3087808.589310443    |9496684.8      |6408876.210689558   |
|2      |llsw3-ethernet-2|2026-10-24 19:14:17.43 |9496684.8        |5485782.643282232    |1.38137692E7   |8327986.556717767   |
|3      |llsw3-ethernet-2|2026-10-24 19:14:18.494|1.38137692E7     |7488605.13010633     |1.89430094E7  

Predicciones auditadas:         320
MAE del modelo en produccion:   1,711,639 B/s   (MAE en la prueba de S10: 113,202 B/s)
MAE de la persistencia en vivo: 1,128,267 B/s


Se borra el grupo del Pushgateway para que Grafana no siga mostrando el último valor como si fuera en vivo.


In [12]:
for sensor in ultima_prediccion:
    urllib.request.urlopen(urllib.request.Request(f"{PUSH_RED}/{sensor}", method="DELETE")).close()
    print("grupo borrado del Pushgateway:", sensor)

grupo borrado del Pushgateway: llsw3-ethernet-2
